# Vectores Semánticos de juguete 🧸
**Idea del paper (Word2Vec):** cada palabra se convierte en una lista de números (un **vector**).

- Palabras parecidas → vectores **cerca**
- Palabras distintas → vectores **lejos**

El paper propone **dos algoritmos** para que la computadora aprenda esos vectores sola leyendo texto:
- **CBOW:** ve las palabras vecinas y adivina la del centro → *"el ___ usa corona"* → **rey**
- **Skip-gram:** ve una palabra y adivina sus vecinas → **rey** → *"usa", "corona"*

Aquí no los entrenamos: escribimos los vectores a mano para ver **qué logran** esos algoritmos.

In [1]:
import numpy as np

## 1. Convertimos palabras en vectores
Cada posición del vector responde una pregunta (1 = sí, 0 = no):

`[¿Es realeza?, ¿Es hombre?, ¿Es mujer?, ¿Es animal?]`

In [2]:
palabras = {
    "rey":    np.array([1, 1, 0, 0]),
    "reina":  np.array([1, 0, 1, 0]),
    "hombre": np.array([0, 1, 0, 0]),
    "mujer":  np.array([0, 0, 1, 0]),
    "perro":  np.array([0, 0, 0, 1]),
}

for palabra, vector in palabras.items():
    print(palabra, vector)

rey [1 1 0 0]
reina [1 0 1 0]
hombre [0 1 0 0]
mujer [0 0 1 0]
perro [0 0 0 1]


## 2. ¿Qué tan cerca están dos palabras? → Distancia euclidiana
Es la distancia "en línea recta" de toda la vida (Pitágoras):
1. Restamos los vectores
2. Elevamos al cuadrado
3. Sumamos
4. Sacamos raíz

**Distancia chica = se parecen. Distancia grande = no se parecen.**

In [3]:
def distancia(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

print("rey vs hombre:", round(distancia(palabras["rey"], palabras["hombre"]), 2))
print("rey vs reina :", round(distancia(palabras["rey"], palabras["reina"]), 2))
print("rey vs perro :", round(distancia(palabras["rey"], palabras["perro"]), 2))

rey vs hombre: 1.0
rey vs reina : 1.41
rey vs perro : 1.73


## 3. Ejemplo 1: Relación semántica (de significado)
Lo más famoso del paper: **rey − hombre + mujer = ?**

In [4]:
resultado = palabras["rey"] - palabras["hombre"] + palabras["mujer"]
print("rey    ", palabras["rey"])
print("- hombre", palabras["hombre"])
print("+ mujer ", palabras["mujer"])
print("= ", resultado)

rey     [1 1 0 0]
- hombre [0 1 0 0]
+ mujer  [0 0 1 0]
=  [1 0 1 0]


## 4. ¿A qué palabra se parece el resultado?
Medimos la distancia del resultado contra cada palabra. La **más chica** gana.

In [5]:
for palabra, vector in palabras.items():
    print(palabra, "->", round(distancia(resultado, vector), 2))
# La distancia más chica es 'reina' (0.0) 👑

rey -> 1.41
reina -> 0.0
hombre -> 1.73
mujer -> 1.0
perro -> 1.73


## 5. Ejemplo 2: Relación sintáctica (de forma de la palabra)
El paper también pregunta: *¿qué palabra es a "pequeño" lo que "grandísimo" es a "grande"?*

**grandísimo − grande + pequeño = ?**

Ahora las preguntas del vector son: `[¿Es grande?, ¿Es pequeño?, ¿Es exagerado (-ísimo)?]`

In [6]:
tamaños = {
    "grande":      np.array([1, 0, 0]),
    "grandísimo":  np.array([1, 0, 1]),
    "pequeño":     np.array([0, 1, 0]),
    "pequeñísimo": np.array([0, 1, 1]),
}

for palabra, vector in tamaños.items():
    print(palabra, vector)

grande [1 0 0]
grandísimo [1 0 1]
pequeño [0 1 0]
pequeñísimo [0 1 1]


In [7]:
resultado2 = tamaños["grandísimo"] - tamaños["grande"] + tamaños["pequeño"]
print("grandísimo", tamaños["grandísimo"])
print("- grande  ", tamaños["grande"])
print("+ pequeño ", tamaños["pequeño"])
print("= ", resultado2, "\n")

for palabra, vector in tamaños.items():
    print(palabra, "->", round(distancia(resultado2, vector), 2))
# La distancia más chica es 'pequeñísimo' (0.0) 🐜

grandísimo [1 0 1]
- grande   [1 0 0]
+ pequeño  [0 1 0]
=  [0 1 1] 

grande -> 1.73
grandísimo -> 1.41
pequeño -> 1.0
pequeñísimo -> 0.0


## Resumen
1. Cada palabra es un **vector** (una lista de números).
2. Con la **distancia** sabemos qué palabras se parecen.
3. Podemos **sumar y restar** palabras:
   - Semántica: rey − hombre + mujer = reina
   - Sintáctica: grandísimo − grande + pequeño = pequeñísimo
4. En el paper real, la computadora **aprende sola** los números con **CBOW** o **Skip-gram**, leyendo millones de textos.